# Online Purchase Intention Prediction

## Preprocessing & Modelling

### Purpose of this notebook

This notebook picks up where `01_data_audit_eda.ipynb` left off. It builds a reproducible preprocessing pipeline, establishes baseline models, trains and tunes an XGBoost model, and evaluates performance.

The analysis focuses on:

- Train/test split (stratified, given the ~15.5% class imbalance found in notebook 1)
- Encoding of categorical features
- Dummy Classifier and untuned Random Forest baselines
- Initial XGBoost model with class-imbalance handling
- Model comparison with and without `PageValues` (per the leakage discussion in notebook 1)
- Hyperparameter tuning (Randomized Search → Grid Search → Manual Search)
- Feature importance and business interpretation

### Input

This notebook loads the same raw dataset used in notebook 1 (`../data/raw/online_shoppers_intention.csv`). No rows were removed in notebook 1 (the 125 duplicates were intentionally retained), so the raw file is reloaded here directly.

## 1. Imports & Setup

In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

SEED = 1

## 2. Load Data

In [2]:
shopping_df = pd.read_csv("../data/raw/online_shoppers_intention.csv")
shopping_df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


## 3. Define X / y

In [3]:
X = shopping_df.drop("Revenue", axis=1)
y = shopping_df["Revenue"]

# Confirm class balance (should match notebook 1: ~84.5% / ~15.5%)
y.value_counts(normalize=True)

Revenue
False    0.845255
True     0.154745
Name: proportion, dtype: float64

## 4. Train / Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

X_train shape: (9864, 17)
X_test shape:  (2466, 17)


In [5]:
# Verify stratify worked as expected
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

Revenue
False    0.845296
True     0.154704
Name: proportion, dtype: float64
Revenue
False    0.845093
True     0.154907
Name: proportion, dtype: float64


## 5. Categorical vs Numerical Columns

`OperatingSystems`, `Browser`, `Region`, and `TrafficType` are stored as `int64` but are actually category codes (no meaningful order), so they're converted to `str` before splitting columns by dtype.

In [6]:
cols_to_convert = ['OperatingSystems', 'Browser', 'Region', 'TrafficType']
X_train[cols_to_convert] = X_train[cols_to_convert].astype(str)
X_test[cols_to_convert] = X_test[cols_to_convert].astype(str)

numerical_cols = X_train.select_dtypes(include='number').columns
categorical_cols = X_train.select_dtypes(include='object').columns

print(numerical_cols)
print(categorical_cols)

Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'],
      dtype='object')
Index(['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType',
       'VisitorType'],
      dtype='object')


In [7]:
# Check cardinality before encoding
for col in categorical_cols:
    print(col, X_train[col].nunique())

Month 10
OperatingSystems 8
Browser 13
Region 9
TrafficType 20
VisitorType 3


## 6. One-Hot Encoding

In [8]:
encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')

X_train_cat_encoded = encoder.fit_transform(X_train[categorical_cols])
X_test_cat_encoded = encoder.transform(X_test[categorical_cols])

encoded_cols = encoder.get_feature_names_out(categorical_cols)
X_train_cat_df = pd.DataFrame(X_train_cat_encoded, columns=encoded_cols, index=X_train.index)
X_test_cat_df = pd.DataFrame(X_test_cat_encoded, columns=encoded_cols, index=X_test.index)

X_train_cat_df.head()

,Month_Dec,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep,OperatingSystems_2,...,TrafficType_20,TrafficType_3,TrafficType_4,TrafficType_5,TrafficType_6,TrafficType_7,TrafficType_8,TrafficType_9,VisitorType_Other,VisitorType_Returning_Visitor
7349,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
8611,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3877,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2625,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3508,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 7. Combine Numerical + Encoded Categorical Features

In [9]:
X_train_final = pd.concat([X_train[numerical_cols], X_train_cat_df], axis=1)
X_test_final = pd.concat([X_test[numerical_cols], X_test_cat_df], axis=1)

print(X_train_final.shape)
print(X_test_final.shape)

(9864, 67)
(2466, 67)


## 8. Baseline Models

Two reference points before XGBoost:
1. **Dummy Classifier** — always predicts the majority class, to confirm any real model must beat this.
2. **Untuned Random Forest** — a reasonable non-boosted baseline.

Primary metric: **PR-AUC** (average precision), consistent with the project plan, since accuracy is misleading on this imbalanced target.

In [10]:
dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy.fit(X_train_final, y_train)
y_pred_dummy = dummy.predict(X_test_final)

print(classification_report(y_test, y_pred_dummy))

              precision    recall  f1-score   support

       False       0.85      1.00      0.92      2084
        True       0.00      0.00      0.00       382

    accuracy                           0.85      2466
   macro avg       0.42      0.50      0.46      2466
weighted avg       0.71      0.85      0.77      2466



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [11]:
rf = RandomForestClassifier(random_state=SEED)
rf.fit(X_train_final, y_train)
y_pred_rf = rf.predict(X_test_final)

print(classification_report(y_test, y_pred_rf))
print("PR-AUC:", average_precision_score(y_test, rf.predict_proba(X_test_final)[:, 1]))

              precision    recall  f1-score   support

       False       0.92      0.97      0.95      2084
        True       0.78      0.55      0.65       382

    accuracy                           0.91      2466
   macro avg       0.85      0.76      0.80      2466
weighted avg       0.90      0.91      0.90      2466

PR-AUC: 0.7623716334199628


## 9. XGBoost — Initial Model

In [16]:
xgb = XGBClassifier(random_state=SEED, eval_metric='logloss')
xgb.fit(X_train_final, y_train)
y_pred_xgb = xgb.predict(X_test_final)

print(classification_report(y_test, y_pred_xgb))
print("PR-AUC:", average_precision_score(y_test, xgb.predict_proba(X_test_final)[:, 1]))

              precision    recall  f1-score   support

       False       0.93      0.96      0.94      2084
        True       0.73      0.59      0.65       382

    accuracy                           0.90      2466
   macro avg       0.83      0.77      0.80      2466
weighted avg       0.90      0.90      0.90      2466

PR-AUC: 0.748552233194997


In [17]:
scale_pos_weight = (y_train == False).sum() / (y_train == True).sum()
print("scale_pos_weight:", scale_pos_weight)

xgb_weighted = XGBClassifier(random_state=SEED, eval_metric='logloss', 
                               scale_pos_weight=scale_pos_weight)
xgb_weighted.fit(X_train_final, y_train)
y_pred_xgb_weighted = xgb_weighted.predict(X_test_final)

print(classification_report(y_test, y_pred_xgb_weighted))
print("PR-AUC:", average_precision_score(y_test, xgb_weighted.predict_proba(X_test_final)[:, 1]))

scale_pos_weight: 5.463958060288335
              precision    recall  f1-score   support

       False       0.94      0.91      0.93      2084
        True       0.59      0.71      0.65       382

    accuracy                           0.88      2466
   macro avg       0.77      0.81      0.79      2466
weighted avg       0.89      0.88      0.88      2466

PR-AUC: 0.744221166969977
